# Lab 11: Cascadia Slow Slip Events

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/11_slow_slip_lab.ipynb)


## Introduction

Episodic Tremor and Slip (ETS) events are recurring slow earthquakes on the deep
Cascadia subduction interface, typically at 25–45 km depth — well below the locked
seismogenic zone that will rupture in the next great Cascadia earthquake. During an ETS
the fault creeps by 1–5 cm over days to weeks, too slowly to radiate ordinary seismic
waves but fast enough to generate weak, non-impulsive seismic tremor and to shift GNSS
stations by a few millimetres.

In this lab you will work through the full observational pipeline for the **August–
September 2024** northern Cascadia ETS event: detecting the event in the seismic tremor
record, extracting GNSS displacement offsets across the network, and building the data
vector that will be used in later parts to invert for the slip distribution on the fault.


## Learning objectives

By the end of this lab, you will be able to:

**Part I — Tremor**
- identify an ETS event in a tremor catalog by its elevated event rate
- place the 2024 event in context of the northern Cascadia recurrence record
- estimate event start and end dates from seismic tremor data

**Part II — GNSS offsets**
- remove secular and seasonal motion from a GNSS time series
- measure pre/post-event displacement offsets with propagated uncertainties
- estimate and remove a common-mode signal from a GNSS network
- produce and interpret a horizontal displacement map

**Part III — Green's functions**
- explain what a Green's function represents physically
- build the Green's function matrix G using an elastic half-space solution
- verify the sign and approximate amplitude of G against physical expectations

**Part IV — Slip inversion**
- formulate the regularized weighted least-squares inverse problem
- select regularisation strength λ using the L-curve
- explain why Laplacian smoothing does not constrain total seismic moment
- apply Dirichlet boundary conditions and amplitude damping to stabilise the inversion
- compute seismic moment magnitude and compare to published ETS scaling relations


## Notebook outline

- [Setup](#setup)
- [Part I: Identifying the 2024 ETS event](#part-i-identifying-the-2024-ets-event)
- [Part II: Measuring GNSS offsets](#part-ii-measuring-gnss-offsets)
- [Part III: Fault mesh and Green's functions](#part-iii-fault-mesh-and-greens-functions)
- [Part IV: Slip inversion](#part-iv-slip-inversion)
- [Synthesis](#synthesis)
- [Summary](#summary)


## Setup

Run this cell once before continuing. It installs all packages needed for the
full lab and defines shared helper functions used in Parts I and II.


In [ ]:
%pip install -q cartopy requests h5netcdf xarray

import os, io
import numpy as np
import pandas as pd
import requests
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10,
                     'axes.titlesize': 11, 'axes.grid': True, 'grid.alpha': 0.2})

PROJ = ccrs.PlateCarree()

def setup_map(ax, extent):
    """Apply standard Cascadia map features to a cartopy axes."""
    ax.set_extent(extent, crs=PROJ)
    ax.add_feature(cfeature.LAND,      facecolor='0.93',    zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor='#cde8f0', zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=0.8, edgecolor='0.2', zorder=2)
    ax.add_feature(cfeature.BORDERS,   lw=0.6, edgecolor='0.3', linestyle='--', zorder=2)
    ax.add_feature(cfeature.STATES,    lw=0.4, edgecolor='0.45', zorder=2)
    gl = ax.gridlines(draw_labels=True, lw=0.3, color='gray', alpha=0.5)
    gl.top_labels = False; gl.right_labels = False

print('Setup complete.')


---

## Part I: Identifying the 2024 ETS event

### Prediction

The tremor catalog covers all of northern Cascadia (46–51°N) from 2020–2025. Before plotting anything, consider:

1. Would you expect tremor detections to be distributed roughly uniformly through time, or concentrated in bursts? What controls the timing?
2. If ETS events recur on a ~14-month cycle, how many distinct elevated-rate periods would you expect to see in a five-year record?

Write your predictions before running the cells below.


### I.1 — Load the tremor catalog


In [ ]:
CSV_URL = ('https://raw.githubusercontent.com/amtseismo/EPS166/'
           '51f43864d41c249a5724c9f4a481fa0ef14ada3b/'
           'datasets/tremor_events-2010-2025.csv')

print('Loading tremor catalog …')
tremor = pd.read_csv(CSV_URL, dtype={'duration': str})
tremor.columns = tremor.columns.str.strip()
tremor['time'] = pd.to_datetime(tremor['starttime'])

# Restrict to northern Cascadia, 2020–2025
tremor = (tremor[(tremor['lat'] >= 46) & (tremor['lat'] <= 51) &
                 (tremor['time'].dt.year >= 2020) &
                 (tremor['time'].dt.year <= 2025)]
               .sort_values('time')
               .reset_index(drop=True))

print(f'  {len(tremor):,} detections  (46–51°N, 2020–2025)')
print(f'  {tremor["time"].min().date()} – {tremor["time"].max().date()}')
tremor[['time', 'lat', 'lon', 'depth']].head()


### I.2 — Event rate through time

Plot the daily tremor detection count for 2020–2025 alongside a 14-day rolling mean.


In [ ]:
daily = (tremor.set_index('time')
               .resample('1D').size()
               .rename('count').reset_index())
roll  = daily.set_index('time')['count'].rolling('14D', center=True).mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(daily['time'], daily['count'], width=1.0,
       color='steelblue', alpha=0.5, label='Daily detections')
ax.plot(roll.index, roll.values, color='firebrick', lw=2, label='14-day mean')
ax.set_xlabel('Year')
ax.set_ylabel('Tremor detections / day')
ax.set_title('PNSN tremor catalog — Northern Cascadia (46–51°N)  |  2020–2025')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print('Annual detections:')
print(tremor.groupby(tremor['time'].dt.year).size().rename('detections').to_string())


> **Interpret the event-rate record:**
> 1. How many distinct periods of elevated tremor rate (i.e. >300 tremors/day) can you identify in the 2020–2025 record? How does that compare to your prediction?
> 2. Estimate the recurrence interval between events by measuring the time between successive peaks in the rolling mean.
> 3. How does the 2024 event compare to the others in peak rate and duration?
> 4. The background rate between events is non-zero. What does persistent low-level tremor suggest about the fault between ETS episodes?


### I.3 — Spatial distribution of the 2024 event

Zoom in on the 2024 event and map tremor locations coloured by time. Adjust `PLOT_START` and `PLOT_END` to bracket the elevated-rate period you identified above.


In [ ]:
PLOT_START = '2024-07-15'
PLOT_END   = '2024-10-15'

sub   = tremor[(tremor['time'] >= PLOT_START) & (tremor['time'] <= PLOT_END)].copy()
t_num = (sub['time'] - sub['time'].min()).dt.total_seconds().values

fig, ax = plt.subplots(figsize=(7, 9), subplot_kw={'projection': PROJ})
setup_map(ax, extent=[-128, -120, 45.5, 51.5])
sc = ax.scatter(sub['lon'], sub['lat'], c=t_num, cmap='plasma',
                s=2, alpha=0.5, zorder=3, transform=PROJ)

tick_vals = np.linspace(t_num.min(), t_num.max(), 5)
tick_labs = [(sub['time'].min() + pd.Timedelta(seconds=float(v))).strftime('%b %d')
             for v in tick_vals]
cb = plt.colorbar(sc, ax=ax, orientation='vertical', shrink=0.55, pad=0.03)
cb.set_ticks(tick_vals); cb.set_ticklabels(tick_labs)
cb.set_label('Date', fontsize=10)
ax.set_title(f'2024 tremor locations coloured by time\n{PLOT_START} – {PLOT_END}',
             fontsize=11)
plt.tight_layout(); plt.show()
print(f'{len(sub):,} detections in window')


> **Interpret the spatial distribution:**
> 1. Estimate the along-strike length and trench-normal width of the tremor zone.
> 2. Using the time coloring, describe any apparent spatial progression of tremor during the event. Is there a clear propagation direction, or does the pattern look more complex?
> 3. How does the depth range of ETS (25–45 km on the Cascadia interface) relate to what you can infer from the map geometry?


### I.4 — Determine the event dates

Based on the event-rate plot, enter your best estimates for when the 2024 ETS began and ended. The start is typically easier to define than the end — explain in your interpretation how you chose each date.


In [ ]:
# ── Enter your estimated ETS dates ──────────────────────────────────────────
SSE_START = '2024-07-01'   # replace with your start date, e.g. 'YYYY-MM-DD'
SSE_END   = '2024-10-01'   # replace with your end date,   e.g. 'YYYY-MM-DD'

# --- do not edit below this line ---
SSE_START = pd.Timestamp(SSE_START)
SSE_END   = pd.Timestamp(SSE_END)
duration  = (SSE_END - SSE_START).days
print(f'SSE_START : {SSE_START.date()}')
print(f'SSE_END   : {SSE_END.date()}')
print(f'Duration  : {duration} days')
if duration < 5 or duration > 90:
    print('\n⚠ Check your dates — that duration looks unusual for a Cascadia ETS.')


In [ ]:
# ── Verify: overlay your window on the full event-rate plot ──────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(daily['time'], daily['count'], width=1.0,
       color='steelblue', alpha=0.5, label='Daily detections')
ax.plot(roll.index, roll.values, color='firebrick', lw=2, label='14-day mean')
ax.axvspan(SSE_START, SSE_END, color='gold', alpha=0.4,
           label=f'Your window  '
                 f'({SSE_START.strftime("%b %d")} – {SSE_END.strftime("%b %d")})')
ax.axvline(SSE_START, color='darkorange', lw=1.5, ls='--')
ax.axvline(SSE_END,   color='darkorange', lw=1.5, ls='--')
n_window = int(((tremor['time'] >= SSE_START) & (tremor['time'] < SSE_END)).sum())
ax.set_title(f'SSE window: {SSE_START.date()} – {SSE_END.date()}  '
             f'({duration} days, {n_window:,} tremor detections)')
ax.set_xlabel('Year')
ax.set_ylabel('Tremor detections / day')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


> **Interpret the event dates:**
> 1. How precisely can you determine the start date from the event-rate plot alone? What contributes to the uncertainty?


---

## Part II: Measuring GNSS offsets

### Prediction

Before loading any data, consider what a slow-slip event on the deep Cascadia
thrust interface should look like in GNSS:

1. During an ETS, the fault creeps in the updip direction. How should coastal
   GNSS stations on the overriding plate move — toward or away from the trench?
   In which compass direction?
2. How should the signal vary with distance from the fault? Should inland stations
   see larger or smaller offsets than coastal ones?
3. The pre- and post-event means are each computed over a finite window. What
   would happen to the estimated offset if the window were very short (2 days)?
   Very long (90 days)?

Write your predictions before running any cells.


### II.1 — Load the GNSS dataset

The GNSS dataset contains daily east, north, and up positions (in metres,
relative to a reference position) together with their daily uncertainties,
for stations across the Pacific Northwest. It is stored as a NetCDF file
and accessed with `xarray`.


In [ ]:
# ── Download and cache GNSS dataset ──────────────────────────────────────────
GNSS_URL = ('https://raw.githubusercontent.com/amtseismo/EPS166/'
            '51f43864d41c249a5724c9f4a481fa0ef14ada3b/'
            'datasets/gnss_SOPAC_trend_2010_2025.nc')
LOCAL = 'gnss_SOPAC_trend_2010_2025.nc'

if not os.path.exists(LOCAL):
    print('Downloading GNSS dataset … (this may take a minute)')
    r = requests.get(GNSS_URL, timeout=300); r.raise_for_status()
    with open(LOCAL, 'wb') as f: f.write(r.content)
    print('  saved to', LOCAL)

ds = xr.open_dataset(LOCAL, engine='h5netcdf')
times_all = pd.to_datetime(ds.time.values)
print(f'{ds.sizes["station"]} stations,  '
      f'{times_all.min().date()} – {times_all.max().date()}')
print('Variables:', list(ds.data_vars))


### II.2 — Select stations

Define the geographic bounding box for the ETS region. Use the displacement
map from Part I (or the tremor map from Part I) to judge appropriate limits.
The cell also identifies a set of **reference stations** south of the ETS
zone; these will be used for the common-mode filter in II.5.


In [ ]:
# ── Define bounds — adjust to match your tremor map from Part I ───────────────
LAT_MIN, LAT_MAX = 45.5, 50.0   # °N
LON_MIN, LON_MAX = -127.0, -121.5   # °E (negative = west)

# Reference stations for the common-mode filter.
# Stations more than CMF_DIST_KM from the ETS centroid should not be moving
# due to slow slip and can be used to estimate network-wide noise.
# This matches the criterion used in the slip-inversion script.
EVENT_LAT   = 48.5    # approximate ETS centroid (°N)
EVENT_LON   = -124.5  # approximate ETS centroid (°E)
CMF_DIST_KM = 600.0   # reference stations must be at least this far away

def haversine_km(lat, lon):
    """Great-circle distance (km) from each station to the ETS centroid."""
    R = 6371.0
    dlat = np.radians(np.asarray(lat) - EVENT_LAT)
    dlon = np.radians(np.asarray(lon) - EVENT_LON)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(EVENT_LAT)) * np.cos(np.radians(lat))
         * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(a))

all_lats  = ds.lat.values
all_lons  = ds.lon.values
all_names = [str(s) for s in ds.station.values]
dists_km  = haversine_km(all_lats, all_lons)

in_box  = ((all_lats >= LAT_MIN) & (all_lats <= LAT_MAX) &
           (all_lons >= LON_MIN) & (all_lons <= LON_MAX))
is_ref  = dists_km >= CMF_DIST_KM

ana_stations = [n for n, m in zip(all_names, in_box) if m]
ref_stations = [n for n, m in zip(all_names, is_ref) if m]
print(f'Analysis stations (ETS region):                {len(ana_stations)}')
print(f'CMF reference stations (≥{CMF_DIST_KM:.0f} km from event): {len(ref_stations)}')

# ── Station map ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': PROJ})
setup_map(ax, extent=[-130, -119, 43, 51.5])

# Reference-only stations (not in analysis box)
ref_only = is_ref & ~in_box
ax.scatter(all_lons[in_box],   all_lats[in_box],   s=25, color='steelblue',
           zorder=4, transform=PROJ, label=f'Analysis ({in_box.sum()})')
ax.scatter(all_lons[ref_only], all_lats[ref_only], s=20, color='firebrick',
           zorder=4, transform=PROJ, label=f'CMF reference ({is_ref.sum()})',
           marker='s')
ax.scatter(all_lons[~in_box & ~is_ref], all_lats[~in_box & ~is_ref],
           s=8, color='0.7', zorder=2, transform=PROJ, label='Not used')

# ETS centroid
ax.scatter(EVENT_LON, EVENT_LAT, s=120, color='gold', edgecolors='k',
           linewidths=0.8, marker='*', zorder=7, transform=PROJ,
           label='ETS centroid')

# 400 km radius circle (approximate in geographic coordinates)
theta  = np.linspace(0, 2*np.pi, 361)
r_lat  = CMF_DIST_KM / 111.195
r_lon  = CMF_DIST_KM / (111.195 * np.cos(np.radians(EVENT_LAT)))
ax.plot(EVENT_LON + r_lon*np.cos(theta),
        EVENT_LAT + r_lat*np.sin(theta),
        color='firebrick', lw=1.2, ls='--', transform=PROJ,
        label=f'{CMF_DIST_KM:.0f} km radius')

ax.legend(fontsize=9, loc='lower right')
ax.set_title(f'GNSS station selection\n'
             f'CMF reference = stations ≥ {CMF_DIST_KM:.0f} km from ETS centroid',
             fontsize=11)
plt.tight_layout(); plt.show()


> **Interpret the station distribution:**
> 1. How many stations fall within your bounding box? Does the coverage
>    look adequate to constrain slip across the entire ETS zone, or are
>    there spatial gaps?
> 2. The reference stations are those more than 600 km from the ETS centroid.
>    In which directions from the event do reference stations exist —
>    only to the south, or also to the east and north?
> 3. The CMF assumes reference stations are not moving due to slow slip.
>    Is that assumption likely to hold for all of them? Are there any
>    reference stations you would be cautious about using?


### II.3 — Remove secular motion

Each GNSS station records a combination of:
- **Secular velocity** — steady plate motion (~40 mm/yr westward at Cascadia)
- **Seasonal signals** — annual and semiannual oscillations from atmospheric
  loading, snow, and other seasonal processes
- **Transients** — slow slip events and other deformation signals

Before measuring an ETS offset, we must remove the secular and seasonal
components. We fit the model

$$y(t) = a + vt + A_1\cos(2\pi t) + B_1\sin(2\pi t) + A_2\cos(4\pi t) + B_2\sin(4\pi t)$$

to the full time series using least squares, then subtract the prediction.
The residual $y(t) - \hat{y}(t)$ should be near zero except during transient
events like ETS.


In [ ]:
def secular_residual(t_yr, vals_mm):
    """Remove secular velocity + annual + semiannual signal.
    Returns the residual in mm, or None if fewer than 180 valid days."""
    valid = np.isfinite(vals_mm)
    if valid.sum() < 180:
        return None
    t0 = float(t_yr[valid].mean())
    G  = np.column_stack([
        np.ones(len(t_yr)),   t_yr - t0,
        np.cos(2*np.pi*t_yr), np.sin(2*np.pi*t_yr),
        np.cos(4*np.pi*t_yr), np.sin(4*np.pi*t_yr)])
    p, _, _, _ = np.linalg.lstsq(G[valid], vals_mm[valid], rcond=None)
    return vals_mm - G @ p

# ── Show the secular decomposition for one station ────────────────────────────
EXAMPLE_STA = 'albh'   # change to any station in ana_stations

s      = ds.sel(station=EXAMPLE_STA)
times  = pd.to_datetime(s.time.values)
t_yr   = s.dec_year.values
e_mm   = 1000 * s.east_m.values
res_e  = secular_residual(t_yr, e_mm)

# Reconstruct the secular model from the difference
model  = e_mm - res_e

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(times, e_mm,   color='0.6',       lw=0.6, label='Raw east (mm)')
axes[0].plot(times, model,  color='firebrick',  lw=1.5, label='Secular model')
axes[0].set_ylabel('East position (mm)'); axes[0].legend(fontsize=9)
axes[0].set_title(f'{EXAMPLE_STA.upper()} — secular decomposition', fontsize=11)

axes[1].plot(times, res_e, color='steelblue', lw=0.8, label='Residual (mm)')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].axvspan(SSE_START, SSE_END, color='gold', alpha=0.35, label='SSE window')
axes[1].set_ylabel('Residual (mm)'); axes[1].set_xlabel('Year')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()


### II.4 — Measure offsets

The ETS displacement at each station is estimated as the difference between
a **post-event mean** and a **pre-event mean** of the secular residual:

$$\Delta E = \overline{r_E(t_{\text{post}})} - \overline{r_E(t_{\text{pre}})}$$

where the pre-window runs from `SSE_START − WINDOW_DAYS` to `SSE_START`,
and the post-window runs from `SSE_END` to `SSE_END + WINDOW_DAYS`.
The uncertainty is propagated from the daily position sigmas stored in the
dataset, assuming independent daily errors.

**Data-quality filter:** a station is included only if at least 90% of
days in *both* windows have valid observations. GNSS time series have
occasional gaps from power outages, telemetry failures, or antenna problems.
A short or gappy window produces an unreliable mean, so affected stations
are dropped. This is why the number of stations passing through
`measure_offsets` is smaller than the number selected in II.2.


In [ ]:
def measure_offsets(ds, station_list, SSE_START, SSE_END, window_days=14):
    """Compute east and north offsets for every station in station_list.
    Returns {station: {de, dn, se, sn, lat, lon}}.
    Stations with < 90 % data coverage in either window are skipped."""
    pre_start = SSE_START - pd.Timedelta(days=window_days)
    post_end  = SSE_END   + pd.Timedelta(days=window_days)
    min_pts   = int(np.ceil(window_days * 0.9))
    result    = {}
    for sta in station_list:
        try:
            s      = ds.sel(station=sta)
            times  = pd.to_datetime(s.time.values)
            t_yr   = s.dec_year.values
            e_mm   = 1000 * s.east_m.values
            n_mm   = 1000 * s.north_m.values
            se_day = 1000 * s.east_sigma_m.values
            sn_day = 1000 * s.north_sigma_m.values
        except Exception:
            continue
        res_e = secular_residual(t_yr, e_mm)
        res_n = secular_residual(t_yr, n_mm)
        if res_e is None or res_n is None:
            continue
        pre_e  = (times >= pre_start) & (times < SSE_START) & np.isfinite(res_e)
        post_e = (times >= SSE_END)   & (times < post_end)  & np.isfinite(res_e)
        pre_n  = (times >= pre_start) & (times < SSE_START) & np.isfinite(res_n)
        post_n = (times >= SSE_END)   & (times < post_end)  & np.isfinite(res_n)
        if any(m.sum() < min_pts for m in [pre_e, post_e, pre_n, post_n]):
            continue
        de = float(np.nanmean(res_e[post_e]) - np.nanmean(res_e[pre_e]))
        dn = float(np.nanmean(res_n[post_n]) - np.nanmean(res_n[pre_n]))
        se = max(float(np.sqrt(np.nansum(se_day[pre_e]**2) / pre_e.sum()**2 +
                               np.nansum(se_day[post_e]**2) / post_e.sum()**2)), 0.1)
        sn = max(float(np.sqrt(np.nansum(sn_day[pre_n]**2) / pre_n.sum()**2 +
                               np.nansum(sn_day[post_n]**2) / post_n.sum()**2)), 0.1)
        result[sta] = dict(de=de, dn=dn, se=se, sn=sn,
                           lat=float(s.lat.values), lon=float(s.lon.values))
    return result

# ── Compute baseline offsets: 14-day windows, no CMF ─────────────────────────
WINDOW_DAYS = 14
offsets_raw = measure_offsets(ds, ana_stations, SSE_START, SSE_END,
                               window_days=WINDOW_DAYS)
print(f'{len(offsets_raw)} stations with sufficient data coverage')

# Summary table
rows = []
for sta, v in sorted(offsets_raw.items(), key=lambda x: -abs(x[1]['de'])):
    rows.append({'Station': sta.upper(),
                 'ΔE (mm)': f"{v['de']:+.2f}", 'σE': f"{v['se']:.2f}",
                 'ΔN (mm)': f"{v['dn']:+.2f}", 'σN': f"{v['sn']:.2f}",
                 'Lat': f"{v['lat']:.2f}", 'Lon': f"{v['lon']:.2f}"})
pd.DataFrame(rows)


In [ ]:
def running_median(arr, half_days=3):
    return pd.Series(arr).rolling(2*half_days+1, center=True, min_periods=1).median().values

# ── Signal-to-noise filter ───────────────────────────────────────────────────
# Keep only stations where the displacement exceeds the formal uncertainty
# in at least one component.
MIN_SNR = 1.0
offsets_plot = {sta: v for sta, v in offsets_raw.items()
                if max(abs(v['de'])/v['se'],
                       abs(v['dn'])/v['sn']) >= MIN_SNR}
print(f'Plotting {len(offsets_plot)} of {len(offsets_raw)} stations '
      f'(SNR \u2265 {MIN_SNR} in at least one component)')


# ── Figure A: Displacement map (replicates panel 2 of the inversion figure) ──
lons = np.array([v['lon'] for v in offsets_plot.values()])
lats = np.array([v['lat'] for v in offsets_plot.values()])
des  = np.array([v['de']  for v in offsets_plot.values()])
dns  = np.array([v['dn']  for v in offsets_plot.values()])
mags = np.hypot(des, dns)

ARROW_SCALE = 0.10   # degrees per mm
ext = [lons.min()-0.5, lons.max()+0.5, lats.min()-0.3, lats.max()+0.3]

fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={'projection': PROJ})
setup_map(ax, extent=ext)
q = ax.quiver(lons, lats, des*ARROW_SCALE, dns*ARROW_SCALE,
              mags, cmap='Blues', clim=(0, np.percentile(mags, 95)),
              scale=1, scale_units='xy', angles='xy',
              width=0.006, headwidth=4.5, zorder=5, transform=PROJ)
plt.colorbar(q, ax=ax, shrink=0.6, pad=0.03, label='Displacement (mm)')
ref_mm = 5.0
ax.quiver(ext[0]+0.4, ext[2]+0.4, ref_mm*ARROW_SCALE, 0,
          scale=1, scale_units='xy', angles='xy',
          color='k', width=0.007, headwidth=4.5, zorder=6, transform=PROJ)
ax.text(ext[0]+0.6, ext[2]+0.15, f'{ref_mm:.0f} mm', fontsize=9, transform=PROJ)
top15 = sorted(offsets_plot, key=lambda s: -abs(offsets_plot[s]['de'])/offsets_plot[s]['se'])[:15]
for sta in top15:
    v = offsets_plot[sta]
    ax.text(v['lon']+0.08, v['lat'], sta.upper(), fontsize=6.5,
            va='center', zorder=6, transform=PROJ)
ax.set_title(f'GNSS horizontal offsets — SNR \u2265 {MIN_SNR}\n'
             f'{WINDOW_DAYS}-day windows, no CMF', fontsize=11)
plt.tight_layout(); plt.show()

# ── Figure B: Time-series grid for top stations ───────────────────────────────
top_stas = sorted(offsets_raw.keys(),
                  key=lambda s: -abs(offsets_raw[s]['de'])/offsets_raw[s]['se'])[:12]
top_stas = sorted(top_stas, key=lambda s: -offsets_raw[s]['lat'])  # north → south

PLOT_START = SSE_START - pd.Timedelta(days=45)
PLOT_END   = SSE_END   + pd.Timedelta(days=45)
pre_start  = SSE_START - pd.Timedelta(days=WINDOW_DAYS)
post_end   = SSE_END   + pd.Timedelta(days=WINDOW_DAYS)

NCOLS = 4; NROWS = int(np.ceil(len(top_stas) / NCOLS))
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(NCOLS*3.5, NROWS*2.5),
                          sharex=True, sharey=True)
axes = axes.flatten()

for k, sta in enumerate(top_stas):
    ax = axes[k]
    s  = ds.sel(station=sta)
    times = pd.to_datetime(s.time.values)
    res_e = secular_residual(s.dec_year.values, 1000*s.east_m.values)
    res_n = secular_residual(s.dec_year.values, 1000*s.north_m.values)
    if res_e is None:
        ax.set_visible(False); continue
    win   = (times >= PLOT_START) & (times <= PLOT_END)
    t_w   = times[win]
    pre_m = (t_w >= pre_start) & (t_w < SSE_START)
    for arr, color in [(res_e[win], 'steelblue'), (res_n[win], 'darkorange')]:
        ref = np.nanmean(arr[pre_m]) if pre_m.sum() > 0 else 0.0
        a   = arr - ref
        ax.plot(t_w, a, lw=0.5, color=color, alpha=0.25)
        ax.plot(t_w, running_median(a), lw=1.5, color=color)
    ax.axvspan(SSE_START, SSE_END,   color='gold',      alpha=0.30)
    ax.axvspan(pre_start, SSE_START, color='steelblue', alpha=0.08)
    ax.axvspan(SSE_END,   post_end,  color='firebrick', alpha=0.08)
    ax.axhline(0, color='0.5', lw=0.5, ls='--')
    v = offsets_raw[sta]
    ax.set_title(f"{sta.upper()}  {v['lat']:.1f}\N{DEGREE SIGN}N\n"
                 f"dE={v['de']:+.1f}  dN={v['dn']:+.1f} mm", fontsize=8, pad=3)
    ax.set_ylim(-8, 8); ax.tick_params(labelsize=7)
    if k >= (NROWS-1)*NCOLS:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    if k % NCOLS == 0:
        ax.set_ylabel('Residual (mm)', fontsize=8)

for k in range(len(top_stas), len(axes)):
    axes[k].set_visible(False)

fig.suptitle('East (blue) & North (orange) residuals  |  7-day median  |  '
             'Gold = SSE  |  Blue/red = pre/post windows', fontsize=10)
plt.tight_layout(); plt.show()


> **Interpret the offsets and displacement field:**
> 1. What is the sign of the east offset at the coastal stations nearest the
>    fault? Does this match your prediction about the direction of ETS motion?
> 2. Looking at the displacement map, describe how arrow magnitude varies
>    across the network. Which stations move the most, and where are they
>    located relative to the tremor zone you mapped in Part I?
> 3. Do inland stations show smaller offsets than coastal stations? Is the
>    relationship straightforward, or are there exceptions? Suggest a reason
>    for any exception you identify.
> 4. Using the time-series grid, compare the onset sharpness at different
>    stations. Does the ETS appear to begin simultaneously everywhere, or
>    does it arrive at different times across the network?


### II.5 — Common-mode filter

The **common-mode signal** (CME) is a displacement common to all stations in
a network — usually arising from errors in the terrestrial reference frame,
large-scale atmospheric pressure loading, or other non-tectonic effects.
It does not represent deformation at any individual site.

We estimate the CME as the **median** offset across the southern reference
stations identified in II.2. Because those stations are far from the 2024 ETS
and should not be moving due to slow slip, any offset they show is attributed
to common-mode noise. We then subtract that estimate from every analysis station.


In [ ]:
def apply_cme_filter(offsets, ds, ref_stations, SSE_START, SSE_END, window_days=14):
    """Subtract the common-mode signal estimated from ref_stations.
    Returns (corrected_offsets, ref_offsets_dict)."""
    ref = measure_offsets(ds, ref_stations, SSE_START, SSE_END, window_days)
    if not ref:
        print('  No reference stations had sufficient data — CMF skipped.')
        return offsets, {}
    cme_de = float(np.median([v['de'] for v in ref.values()]))
    cme_dn = float(np.median([v['dn'] for v in ref.values()]))
    print(f'  CME estimate ({len(ref)} reference stations):  '
          f'\u0394E = {cme_de:+.2f} mm,  \u0394N = {cme_dn:+.2f} mm')
    corrected = {sta: {**v, 'de': v['de']-cme_de, 'dn': v['dn']-cme_dn}
                 for sta, v in offsets.items()}
    return corrected, ref

offsets_cme, ref_offsets = apply_cme_filter(
    offsets_raw, ds, ref_stations, SSE_START, SSE_END, window_days=WINDOW_DAYS)

cme_de = np.median([v['de'] for v in ref_offsets.values()])
cme_dn = np.median([v['dn'] for v in ref_offsets.values()])

# ── Figure A: The common-mode filter itself ───────────────────────────────────
# Each reference station is far from the ETS zone and should not be moving
# due to slow slip. Any offset they show is treated as network-wide noise.
# The median of their offsets (the CME, red arrow) is what gets subtracted
# from every analysis station.

ref_names = list(ref_offsets.keys())
ref_lons  = np.array([ref_offsets[s]['lon'] for s in ref_names])
ref_lats  = np.array([ref_offsets[s]['lat'] for s in ref_names])
ref_des   = np.array([ref_offsets[s]['de']  for s in ref_names])
ref_dns   = np.array([ref_offsets[s]['dn']  for s in ref_names])
ref_mags  = np.hypot(ref_des, ref_dns)

ana_lons = np.array([offsets_raw[s]['lon'] for s in offsets_raw])
ana_lats = np.array([offsets_raw[s]['lat'] for s in offsets_raw])

# Broad extent to show both reference and analysis regions
all_lons_shown = np.concatenate([ref_lons, ana_lons])
all_lats_shown = np.concatenate([ref_lats, ana_lats])
ext_broad = [all_lons_shown.min()-1, all_lons_shown.max()+1,
             all_lats_shown.min()-0.5, all_lats_shown.max()+0.5]

fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(14, 7),
                                      gridspec_kw={'width_ratios': [1.1, 1]},
                                      subplot_kw={'projection': PROJ},
                                      constrained_layout=True)
# Left: spatial view
ax_map.set_projection = PROJ
# (ax_bar is not a map — rebuild)
plt.close(fig)

fig = plt.figure(figsize=(14, 7), constrained_layout=True)
ax_map = fig.add_subplot(1, 2, 1, projection=PROJ)
ax_bar = fig.add_subplot(1, 2, 2)

setup_map(ax_map, extent=ext_broad)

# Reference station arrows (gray)
q_ref = ax_map.quiver(ref_lons, ref_lats,
                       ref_des * ARROW_SCALE, ref_dns * ARROW_SCALE,
                       ref_mags, cmap='Greys', clim=(0, 3),
                       scale=1, scale_units='xy', angles='xy',
                       width=0.004, headwidth=4, zorder=4, transform=PROJ,
                       label='Reference station offset')

# CME median arrow — plotted at the geographic centre of reference stations
cen_lon = float(np.mean(ref_lons))
cen_lat = float(np.mean(ref_lats))
ax_map.quiver(cen_lon, cen_lat,
              cme_de * ARROW_SCALE, cme_dn * ARROW_SCALE,
              scale=1, scale_units='xy', angles='xy',
              color='firebrick', width=0.009, headwidth=5,
              zorder=6, transform=PROJ,
              label=f'CME median: \u0394E={cme_de:+.2f}, \u0394N={cme_dn:+.2f} mm')

# Mark analysis region
ax_map.scatter(ana_lons, ana_lats, s=12, color='steelblue',
               zorder=5, transform=PROJ, label='Analysis stations')

# Scale bar
ref_mm = 3.0
ax_map.quiver(ext_broad[0]+0.5, ext_broad[2]+0.4,
              ref_mm * ARROW_SCALE, 0,
              scale=1, scale_units='xy', angles='xy',
              color='k', width=0.006, headwidth=4, zorder=7, transform=PROJ)
ax_map.text(ext_broad[0]+0.7, ext_broad[2]+0.15,
            f'{ref_mm:.0f} mm', fontsize=9, transform=PROJ)

ax_map.legend(fontsize=8, loc='upper right')
ax_map.set_title('Reference station offsets\n'
                 'Red arrow = CME (subtracted from all stations)', fontsize=10)

# Right: bar chart of reference station E and N offsets
order = np.argsort(ref_lats)[::-1]   # north → south
x     = np.arange(len(order))
w     = 0.4
ax_bar.bar(x - w/2, ref_des[order], w, color='steelblue', alpha=0.75, label='\u0394E')
ax_bar.bar(x + w/2, ref_dns[order], w, color='darkorange', alpha=0.75, label='\u0394N')
ax_bar.axhline(cme_de, color='steelblue', lw=2.5, ls='--',
               label=f'CME \u0394E = {cme_de:+.2f} mm')
ax_bar.axhline(cme_dn, color='darkorange', lw=2.5, ls='--',
               label=f'CME \u0394N = {cme_dn:+.2f} mm')
ax_bar.axhline(0, color='k', lw=0.5)
ax_bar.set_xticks(x)
ax_bar.set_xticklabels([ref_names[i].upper() for i in order],
                        rotation=45, ha='right', fontsize=7)
ax_bar.set_ylabel('Offset (mm)')
ax_bar.set_title('Individual reference station offsets\n'
                 'Dashed lines = CME estimate (median)', fontsize=10)
ax_bar.legend(fontsize=8)
ax_bar.grid(True, alpha=0.2)

plt.suptitle('Common-mode filter — estimating and removing network-wide noise',
             fontsize=11)
plt.show()

# ── Figure B: Effect of CMF on analysis station offsets ───────────────────────
top = sorted(offsets_raw, key=lambda s: -abs(offsets_raw[s]['de']))[:12]
x   = np.arange(len(top)); w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, comp, key, lbl in zip(axes, ['East', 'North'], ['de', 'dn'],
                               ['\u0394E (mm)', '\u0394N (mm)']):
    raw = [offsets_raw[s][key] for s in top]
    cme = [offsets_cme[s][key] for s in top]
    ax.bar(x - w/2, raw, w, label='No CMF',   color='steelblue', alpha=0.8)
    ax.bar(x + w/2, cme, w, label='With CMF', color='firebrick', alpha=0.8)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([s.upper() for s in top],
                        rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(lbl)
    ax.set_title(f'{comp} offset: before and after CMF')
    ax.legend(fontsize=9)
plt.suptitle('Effect of CMF on analysis station offsets', fontsize=11)
plt.show()


> **Interpret the CMF comparison:**
> 1. How large is the CME estimate in east and north (in mm)? What fraction
>    of the typical offset at a near-field station does it represent?
> 2. At which stations does the CMF make the largest difference? Can you
>    explain why?
> 3. The CMF increases the apparent westward motion at most stations. What
>    does this imply about the sign of the reference-frame bias in this dataset?
> 4. Would you trust the CMF estimate more or less if there were only two
>    reference stations instead of several? Why?


---

## Part III: Fault Mesh and Green's Functions

To invert for slip we need to know how each fault patch would move the surface
for 1 m of slip — the **Green's function**. We compute these numerically using
an elastic half-space solution (cutde), then do a quick forward-model check to
confirm the sign and amplitude are physically reasonable before proceeding to
the inversion.

### Prediction

For updip (thrust) slip on the deep Cascadia interface, in which direction should
coastal GNSS stations move? Should the Green's function amplitude be larger for
shallow patches or deep patches, and why?


In [ ]:
%pip install -q cutde

# ── Helper functions (copied from the slip-inversion script) ──────────────────

# Projection constants
REF_LON, REF_LAT = -124.5, 48.0
SCALE            = 111_195.0   # metres per degree latitude
NU               = 0.25        # Poisson's ratio
ARROW_SCALE      = 0.10        # degrees per mm (for quiver plots)

def lonlat_to_m(lon, lat):
    """Convert lon/lat (degrees) to local Cartesian metres."""
    x = (np.asarray(lon) - REF_LON) * SCALE * np.cos(np.radians(REF_LAT))
    y = (np.asarray(lat) - REF_LAT) * SCALE
    return x, y

def parse_gmsh41(filepath):
    """Parse a Gmsh 4.1 .msh file; return (coords, tri_idx).
    coords: (N_nodes, 3) — lon, lat, depth_km
    tri_idx: (N_elements, 3) — zero-based node indices"""
    with open(filepath) as f:
        lines = f.readlines()
    def sec(tag):
        return next(i for i, l in enumerate(lines) if l.strip() == tag)
    ns      = sec('$Nodes')
    n_nodes = int(lines[ns+1].split()[1])
    coords  = np.array([list(map(float, lines[ns+3+n_nodes+i].split()))
                        for i in range(n_nodes)])
    es      = sec('$Elements')
    n_elems = int(lines[es+1].split()[1])
    tri_idx = np.array([[int(x)-1 for x in lines[es+3+i].split()[1:4]]
                        for i in range(n_elems)], dtype=np.int32)
    return coords, tri_idx

def find_boundary_patches(tri_idx):
    """Boolean mask: True where a patch has at least one unshared edge."""
    from collections import Counter
    edge_count = Counter()
    for tri in tri_idx:
        for i in range(3):
            edge = tuple(sorted((int(tri[i]), int(tri[(i+1)%3]))))
            edge_count[edge] += 1
    bnd = np.zeros(len(tri_idx), dtype=bool)
    for k, tri in enumerate(tri_idx):
        for i in range(3):
            if edge_count[tuple(sorted((int(tri[i]), int(tri[(i+1)%3]))))] == 1:
                bnd[k] = True; break
    return bnd

def tri_area_m2(verts_m):
    e1 = verts_m[1] - verts_m[0]; e2 = verts_m[2] - verts_m[0]
    return 0.5 * float(np.linalg.norm(np.cross(e1, e2)))

print('Part III helpers loaded.')


### III.1 — Load and visualize the fault mesh

The fault mesh discretises the Cascadia subduction interface into triangular
patches. It was built from the McCrory et al. (2012) interface geometry and
covers 46.5–49.5°N at depths of 20–65 km — the full ETS zone and surrounding
region.


In [ ]:
MESH_URL   = ('https://raw.githubusercontent.com/amtseismo/EPS166/'
              'f64ad3d3d3992175a7b92e0435896fc4a842ff25/'
              'datasets/mccrory_ets_45.5_49.5_dep20_65_s3a135.msh')
MESH_LOCAL = 'mccrory_ets_45.5_49.5_dep20_65_s3a135.msh'

# Delete the old 'ets_mesh.msh' if it exists — it was a stale cache.
if not os.path.exists(MESH_LOCAL):
    print('Downloading mesh …')
    r = requests.get(MESH_URL, timeout=60); r.raise_for_status()
    with open(MESH_LOCAL, 'wb') as f: f.write(r.content)

coords, tri_idx = parse_gmsh41(MESH_LOCAL)

# Patch centroids and depth
verts3d   = coords[tri_idx]
centroids = verts3d.mean(axis=1)
patch_lon = centroids[:, 0]
patch_lat = centroids[:, 1]
depth_km  = -centroids[:, 2]          # positive downward (mesh stores z as –km)

bnd_mask = find_boundary_patches(tri_idx)
int_mask = ~bnd_mask
N_PATCHES, N_INT = len(tri_idx), int(int_mask.sum())

print(f'Patches : {N_PATCHES} total  |  {N_INT} interior  |  {bnd_mask.sum()} boundary')
print(f'Depth   : {depth_km.min():.1f} – {depth_km.max():.1f} km')

# ── Mesh map (replicates panel 1 of the inversion displacement figure) ────────
ext = [patch_lon.min()-0.5, patch_lon.max()+0.5,
       patch_lat.min()-0.3, patch_lat.max()+0.3]

fig, ax = plt.subplots(figsize=(7, 9), subplot_kw={'projection': PROJ})
setup_map(ax, extent=ext)

# Colour-filled mesh (depth) — PlateCarree GeoAxes: data coords are lon/lat
tc = ax.tripcolor(coords[:,0], coords[:,1], tri_idx,
                  facecolors=depth_km, cmap='plasma_r',
                  vmin=20, vmax=65, zorder=3, alpha=0.85)
ax.triplot(coords[:,0], coords[:,1], tri_idx,
           color='0.55', lw=0.2, zorder=4, alpha=0.5)
plt.colorbar(tc, ax=ax, shrink=0.6, pad=0.03).set_label('Depth (km)', fontsize=10)

# GNSS stations (from Part II offsets)
sta_list = list(offsets_raw.keys())
lons_sta = np.array([offsets_raw[s]['lon'] for s in sta_list])
lats_sta = np.array([offsets_raw[s]['lat'] for s in sta_list])
ax.scatter(lons_sta, lats_sta, s=20, color='k', marker='^',
           zorder=5, transform=PROJ, label='GNSS stations')
ax.legend(fontsize=9, loc='lower right')
ax.set_title(f'Cascadia subduction interface\n'
             f'{N_PATCHES} patches, 20–65 km depth', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()


### III.2 — Build the Green's function matrix

Each column of **G** is the predicted surface displacement field (east and north,
in mm) for 1 m of updip dip-slip on one interior fault patch, computed using
the cutde elastic half-space solution. The matrix has shape
(2 × N_stations) × N_interior_patches.

**Boundary patches are excluded** — their slip is fixed at zero by construction,
so they contribute no columns to G.


In [ ]:
try:
    import cutde.halfspace as hs
except ImportError:
    raise ImportError('cutde not found. Run: pip install cutde')

N_STA = len(sta_list)
M_OBS = 2 * N_STA

# Observation points in Cartesian metres (z = 0: surface)
obs_x, obs_y = lonlat_to_m(lons_sta, lats_sta)
obs_xyz = np.column_stack([obs_x, obs_y, np.zeros(N_STA)])

# Mesh vertices in Cartesian metres
node_x, node_y = lonlat_to_m(coords[:,0], coords[:,1])
node_z   = coords[:,2] * 1000.0          # km → m (negative below surface)
node_xyz = np.column_stack([node_x, node_y, node_z])
tris_m   = node_xyz[tri_idx]             # (N_PATCHES, 3, 3) in metres

# Interior patches only
int_indices = np.where(int_mask)[0]
tris_int    = tris_m[int_indices]        # (N_INT, 3, 3)

print(f'Building G ({M_OBS} obs × {N_INT} interior patches) …')
G = np.zeros((M_OBS, N_INT))

try:
    for k in range(N_INT):
        # hs.disp is pairwise: disp[i] = displacement at obs_xyz[i] from tri_rep[i].
        # Tiling one patch gives each station its own copy of the same source triangle.
        tri_rep  = np.tile(tris_int[k], (N_STA, 1, 1))   # (N_STA, 3, 3)
        slip_rep = np.tile([0., +1., 0.], (N_STA, 1))     # updip dip-slip
        disp     = hs.disp(obs_xyz, tri_rep, slip_rep, nu=NU)
        G[0::2, k] = disp[:, 0] * 1000   # east  (mm per m of slip)
        G[1::2, k] = disp[:, 1] * 1000   # north (mm per m of slip)
        if (k+1) % 50 == 0:
            print(f'  {k+1}/{N_INT}', end=' ', flush=True)
    print(f'\n  done.  max|G| = {np.abs(G).max():.2f} mm/m')

except Exception as e:
    if any(s in str(e) for s in ('clBuildProgram', 'PLATFORM_NOT_FOUND',
                                   'no CL platforms', 'BUILD_PROGRAM_FAILURE')):
        print('\nOpenCL error (common on macOS with cutde ≥ 0.3).')
        print("Fix: pip install 'cutde<0.3' then restart the kernel.")
    else:
        raise

# ── Sign check: updip slip → coastal stations move westward (negative east) ───
ref_sta = min(sta_list,
              key=lambda s: abs(offsets_raw[s]['lat']-48.4)+abs(offsets_raw[s]['lon']+123.5))
ref_i   = sta_list.index(ref_sta)
g_east_ref = G[2*ref_i, np.argmax(np.abs(G[2*ref_i, :]))]
sign_ok = g_east_ref < 0
print(f'Sign check ({ref_sta.upper()}): east G = {g_east_ref:.3f} mm/m '
      f'— {"✓ westward (correct)" if sign_ok else "⚠ eastward — check slip_rep sign"}')


### III.3 — Forward model check

Before inverting, verify that the Green's functions behave physically. Compute
the predicted surface displacement for **1 cm of uniform updip slip on all
interior patches** and plot it alongside the observed offsets from Part II.
The two should look qualitatively similar — same general direction and rough
spatial pattern — even though uniform slip is a crude approximation.


In [ ]:
# Predicted displacements: 1 cm = 0.01 m of uniform slip on all interior patches
d_fwd   = G @ (0.01 * np.ones(N_INT))  # mm
de_fwd  = d_fwd[0::2]
dn_fwd  = d_fwd[1::2]
mags_fwd = np.hypot(de_fwd, dn_fwd)

des_obs = np.array([offsets_raw[s]['de'] for s in sta_list])
dns_obs = np.array([offsets_raw[s]['dn'] for s in sta_list])
mags_obs = np.hypot(des_obs, dns_obs)

fig, axes = plt.subplots(1, 2, figsize=(14, 9),
                          subplot_kw={'projection': PROJ},
                          constrained_layout=True)

titles  = ['Observed GNSS offsets\n(from Part II)',
           'Predicted: 1 cm uniform slip\n(forward model)']
de_list = [des_obs, de_fwd]
dn_list = [dns_obs, dn_fwd]
mag_list= [mags_obs, mags_fwd]

for ax, title, de, dn, mags in zip(axes, titles, de_list, dn_list, mag_list):
    setup_map(ax, extent=ext)
    ax.triplot(coords[:,0], coords[:,1], tri_idx,
               color='0.75', lw=0.15, zorder=2, alpha=0.5)
    q = ax.quiver(lons_sta, lats_sta,
                  de * ARROW_SCALE, dn * ARROW_SCALE,
                  mags, cmap='Blues', clim=(0, np.percentile(mags_obs, 95)),
                  scale=1, scale_units='xy', angles='xy',
                  width=0.006, headwidth=4.5, zorder=5, transform=PROJ)
    # Scale bar
    ref_mm = 5.0
    ax.quiver(ext[0]+0.4, ext[2]+0.4, ref_mm*ARROW_SCALE, 0,
              scale=1, scale_units='xy', angles='xy',
              color='k', width=0.007, headwidth=4.5, zorder=6, transform=PROJ)
    ax.text(ext[0]+0.6, ext[2]+0.15, f'{ref_mm:.0f} mm', fontsize=9, transform=PROJ)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Longitude')

fig.colorbar(q, ax=axes, shrink=0.55, pad=0.02, label='Displacement (mm)')
axes[0].set_ylabel('Latitude')
plt.suptitle('Observed vs forward-model displacements', fontsize=12, y=1.01)
plt.show()

print(f'Observed   peak magnitude: {mags_obs.max():.2f} mm')
print(f'Predicted  peak magnitude: {mags_fwd.max():.2f} mm')
print(f'Ratio (obs / pred):        {mags_obs.max()/mags_fwd.max():.2f}')


> **Interpret the forward model:**
> 1. Do the predicted arrows point in the same general direction as the observed
>    ones? Is the spatial pattern (which stations move most) qualitatively similar?
> 2. Where does the uniform-slip forward model agree least with the observations?
>    What does that tell you about where on the fault the actual slip must be
>    concentrated?


---

## Part IV: Slip Inversion

We now have everything we need to invert for the fault slip distribution:
the displacement data vector **d** (from Part II), its uncertainties **σ**,
and the Green's function matrix **G** (from Part III).
The inversion finds the slip vector **m** that best explains the observations
while satisfying a smoothness constraint.

### Prediction

Based on your forward model in Part III, where do you expect the inversion
to concentrate slip? Will the NNLS constraint (m ≥ 0, no back-slip) matter
here — why or why not?


In [ ]:
from scipy.optimize import nnls
from scipy.linalg  import lstsq

MU = 30.0e9   # Pa — shear modulus

def build_spatial_laplacian(centroids_m, radius_m):
    """Graph Laplacian: patches within radius_m are considered neighbours.
    Returns L such that (Lm)_i = k_i m_i - sum_{j in neighbours(i)} m_j,
    where k_i is the number of neighbours of patch i."""
    n     = len(centroids_m)
    dists = np.sqrt(((centroids_m[:,None] - centroids_m[None,:])**2).sum(axis=2))
    adj   = (dists > 0) & (dists <= radius_m)
    return np.diag(adj.sum(axis=1).astype(float)) - adj.astype(float)

def run_lcurve(G, d, sig, lambdas, L, lam_damp=0.0):
    """Scan lambda; return arrays of (NRMS, roughness, slip vectors)."""
    w = 1.0 / sig
    nrms_arr, rough_arr, m_list = [], [], []
    for lam in lambdas:
        rows = [G * w[:,None], lam * L]
        rhs  = [d * w, np.zeros(L.shape[0])]
        if lam_damp > 0:
            rows.append(lam_damp * np.eye(G.shape[1]))
            rhs.append(np.zeros(G.shape[1]))
        Ga  = np.vstack(rows); da = np.concatenate(rhs)
        m,_ = nnls(Ga, da)
        dp  = G @ m
        nrms_arr.append(float(np.sqrt(np.mean(((d - dp) / sig)**2))))
        rough_arr.append(float(np.linalg.norm(L @ m)))
        m_list.append(m)
    return np.array(nrms_arr), np.array(rough_arr), m_list

def compute_mw(m_int, areas_int, mu=MU):
    """Scalar seismic moment and moment magnitude from interior slip vector."""
    M0 = mu * float(np.sum(areas_int * m_int))
    Mw = (2/3) * np.log10(M0) - 6.07 if M0 > 0 else float('nan')
    return M0, Mw

print('Part IV helpers loaded.')


### IV.1 — Assemble the data vector

Stack the east and north offsets for each station into a single column vector
$\mathbf{d}$, and their uncertainties into $\boldsymbol{\sigma}$:

$$\mathbf{d} = \begin{bmatrix} \Delta E_1 \\ \Delta N_1 \\ \Delta E_2 \\ \Delta N_2 \\ \vdots \end{bmatrix},
\qquad
\boldsymbol{\sigma} = \begin{bmatrix} \sigma_{E_1} \\ \sigma_{N_1} \\ \sigma_{E_2} \\ \sigma_{N_2} \\ \vdots \end{bmatrix}$$

The inverse of $\boldsymbol{\sigma}$ defines the data-weighting matrix
$W_d = \mathrm{diag}(1/\sigma_1, 1/\sigma_2, \ldots)$, which down-weights noisy
stations and up-weights well-constrained ones.

We use the CMF-corrected offsets from Part II as the data.


In [ ]:
# Use CMF-corrected offset VALUES but keep the same station list (and order)
# as was used to build G in Part III — a mismatch would corrupt d.
offsets_inv = offsets_cme if 'offsets_cme' in dir() else offsets_raw

# sta_list was built from offsets_raw in Part III; offsets_cme preserves
# the same keys in the same order, so reassigning is safe.
sta_list  = list(offsets_inv.keys())
lons_sta  = np.array([offsets_inv[s]['lon'] for s in sta_list])
lats_sta  = np.array([offsets_inv[s]['lat'] for s in sta_list])
N_STA = len(sta_list)
M_OBS = 2 * N_STA

d_obs = np.zeros(M_OBS)
sig_d = np.zeros(M_OBS)
for i, sta in enumerate(sta_list):
    v = offsets_inv[sta]
    d_obs[2*i],   d_obs[2*i+1]  = v['de'], v['dn']
    sig_d[2*i],   sig_d[2*i+1]  = v['se'], v['sn']

print(f'Data vector:  {M_OBS} observations  ({N_STA} stations × 2 components)')
print(f'  dE range:  {d_obs[0::2].min():.2f} – {d_obs[0::2].max():.2f} mm')
print(f'  dN range:  {d_obs[1::2].min():.2f} – {d_obs[1::2].max():.2f} mm')
print(f'  σ range:   {sig_d.min():.3f} – {sig_d.max():.3f} mm')


#### Aside: data uncertainties and weighting

Each offset $\Delta E_i$, $\Delta N_i$ has a formal uncertainty $\sigma_i$
propagated from the daily position sigmas stored in the GNSS dataset
(computed in Part II). The data-weighting matrix

$$W_d = \mathrm{diag}\!\left(\frac{1}{\sigma_1},\,\frac{1}{\sigma_2},\,\ldots\right)$$

scales each row of the system so that precise stations (small $\sigma$) count
more than noisy ones. The **normalised RMS misfit**

$$\mathrm{NRMS} = \sqrt{\frac{1}{N}\sum_i \left(\frac{d_i - [G\hat{m}]_i}{\sigma_i}\right)^2}$$

measures fit quality in units of the formal uncertainty: NRMS = 1 means
residuals are on average equal to the stated uncertainties. You will see
NRMS plotted on the L-curve horizontal axis — it is the quantity that
decreases as $\lambda$ decreases and the inversion fits the data more tightly.


### IV.2 — Regularized inverse problem

The forward problem is $G\mathbf{m} = \mathbf{d}$. Because the system is
underdetermined (more patches than observations) and noisy, a direct
pseudo-inverse gives physically unreasonable slip. Instead we minimize

$$\hat{\mathbf{m}} = \underset{\mathbf{m}\,\geq\,0}{\mathrm{argmin}}
\;\underbrace{\left\| W_d\left(G\mathbf{m} - \mathbf{d}\right) \right\|^2}_{\text{data misfit}}
\;+\; \underbrace{\lambda^2 \left\| L\mathbf{m} \right\|^2}_{\text{smoothness penalty}}$$

where $W_d = \mathrm{diag}(1/\boldsymbol{\sigma})$ weights data by uncertainty,
$L$ is the graph Laplacian that penalises roughness, and
$\mathbf{m} \geq 0$ enforces non-negative (thrust) slip. The constraint
$\mathbf{m} \geq 0$ is applied by **non-negative least squares** (NNLS).

In augmented-matrix form this is simply

$$\hat{\mathbf{m}} = \underset{\mathbf{m}\,\geq\,0}{\mathrm{argmin}}
\left\| \begin{bmatrix} W_d G \\ \lambda L \end{bmatrix} \mathbf{m}
- \begin{bmatrix} W_d \mathbf{d} \\ \mathbf{0} \end{bmatrix} \right\|^2$$

**The Laplacian** $L$ is the graph Laplacian of the patch adjacency graph:

$$[L]_{ij} = \begin{cases}
k_i & i = j \quad (k_i = \text{number of neighbours of patch }i) \\
-1  & \text{patches }i\text{ and }j\text{ are neighbours} \\
0   & \text{otherwise}
\end{cases}$$
**Choosing $\lambda$ — the L-curve.** Plot $\|L\hat{m}\|$ (roughness) against
NRMS (data misfit) on log-log axes for a range of $\lambda$ values. The curve
has an L-shape: a flat segment where increasing $\lambda$ reduces roughness
without much cost in fit, and a steep segment where further regularisation
significantly degrades the fit. The **corner** of the L — the point of maximum
curvature — is a standard choice for $\lambda$. Examine the plot and adjust
`LAM_OPT` manually if the auto-detected corner looks wrong.


In [ ]:
LAP_RADIUS_KM = 60.0   # patches within this distance are neighbours
LAMBDAS       = np.logspace(-2, 3, 50)

# Interior-patch centroids in Cartesian metres (for Laplacian)
cx_int, cy_int = lonlat_to_m(patch_lon[int_mask], patch_lat[int_mask])
cents_int_m    = np.column_stack([cx_int, cy_int])

# Naive Laplacian: built from interior patches only → satisfies L·1 = 0
L = build_spatial_laplacian(cents_int_m, LAP_RADIUS_KM * 1e3)
print(f'L shape: {L.shape},  '
      f'mean neighbours: {L.diagonal().mean():.1f}')
print(f'L · 1 = 0? {np.allclose(L @ np.ones(N_INT), 0)}')

# ── L-curve scan ─────────────────────────────────────────────────────────────
print('Running L-curve …')
nrms_arr, rough_arr, m_list = run_lcurve(G, d_obs, sig_d, LAMBDAS, L)

# Geometric corner: point furthest from the chord in log-log space
def lcurve_corner(nrms, rough):
    lx = np.log10(nrms); ly = np.log10(rough)
    lx = (lx - lx.min()) / (lx.max() - lx.min() + 1e-30)
    ly = (ly - ly.min()) / (ly.max() - ly.min() + 1e-30)
    dx = lx[-1] - lx[0]; dy = ly[-1] - ly[0]
    dist = np.abs(dy*(lx-lx[0]) - dx*(ly-ly[0])) / (np.hypot(dx, dy) + 1e-30)
    return int(np.argmax(dist))

corner_idx = lcurve_corner(nrms_arr, rough_arr)

# ── Set LAM_OPT ───────────────────────────────────────────────────────────────
# Leave as None to use the geometric corner, or set a float to override.
# Examine the L-curve plot below and adjust if the auto corner looks wrong.
LAM_OPT = None
lam_opt  = LAM_OPT if LAM_OPT is not None else LAMBDAS[corner_idx]
print(f'Geometric corner: λ = {LAMBDAS[corner_idx]:.3f}  (NRMS = {nrms_arr[corner_idx]:.2f})')
print(f'Using:            λ = {lam_opt:.3f}')

# ── L-curve plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

ax = axes[0]
sc = ax.scatter(nrms_arr, rough_arr, c=np.log10(LAMBDAS), cmap='plasma', s=35)
ax.scatter(nrms_arr[corner_idx], rough_arr[corner_idx],
           s=200, c='k', marker='*', zorder=6,
           label=f'Corner  λ = {LAMBDAS[corner_idx]:.2f}')
if LAM_OPT is not None:
    used_idx = np.argmin(np.abs(LAMBDAS - LAM_OPT))
    ax.scatter(nrms_arr[used_idx], rough_arr[used_idx],
               s=120, c='steelblue', marker='D', zorder=7,
               label=f'Manual  λ = {LAM_OPT:.2f}')
plt.colorbar(sc, ax=ax, label='log₁₀(λ)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Normalised RMS misfit (NRMS)')
ax.set_ylabel('Roughness  ‖Lm‖')
ax.set_title('L-curve'); ax.legend(fontsize=9)

ax = axes[1]
ax.semilogx(LAMBDAS, nrms_arr, 'steelblue', lw=2, label='NRMS misfit')
ax2 = ax.twinx()
ax2.semilogx(LAMBDAS, rough_arr, 'firebrick', lw=2, label='Roughness')
ax.axvline(lam_opt, color='k', ls='--', lw=1.5, label=f'λ = {lam_opt:.2f}')
ax.set_xlabel('λ')
ax.set_ylabel('NRMS misfit', color='steelblue')
ax2.set_ylabel('Roughness', color='firebrick')
ax.set_title('Misfit and roughness vs λ  (adjust LAM_OPT above if needed)')
ax.legend(fontsize=8, loc='upper left')
plt.suptitle('L-curve — selecting regularisation strength λ', fontsize=11)
plt.show()


### IV.3 — Solve: standard smoothed inversion

With $\lambda$ chosen, solve the NNLS problem. The solution $\hat{\mathbf{m}}$
gives updip slip (metres) on each interior patch. Compute the
**seismic moment** and **moment magnitude**:

$$M_0 = \mu \sum_k A_k \, m_k, \qquad
M_w = \tfrac{2}{3}\log_{10}M_0 - 6.07$$

where $\mu = 30\,\mathrm{GPa}$, $A_k$ is patch area, and $m_k$ is slip.
Examine the slip map and the depth distribution of moment.
Do the results look physically reasonable?


In [ ]:
# ── Solve ────────────────────────────────────────────────────────────────────
w    = 1.0 / sig_d
Ga   = np.vstack([G * w[:,None], lam_opt * L])
da   = np.concatenate([d_obs * w, np.zeros(N_INT)])
m_naive, _ = nnls(Ga, da)

d_pred_naive = G @ m_naive
nrms_naive   = float(np.sqrt(np.mean(((d_obs - d_pred_naive) / sig_d)**2)))

int_areas = np.array([tri_area_m2(tris_int[k]) for k in range(N_INT)])
M0_naive, Mw_naive = compute_mw(m_naive, int_areas)

print(f'λ = {lam_opt:.3f}')
print(f'NRMS:       {nrms_naive:.2f}')
print(f'Peak slip:  {m_naive.max()*100:.1f} cm')
print(f'M₀:         {M0_naive:.3e} N·m')
print(f'Mw:         {Mw_naive:.2f}')
print(f'Published Cascadia ETS range: Mw 6.0 – 6.5  (Bartlow 2020)')

# ── Slip map ──────────────────────────────────────────────────────────────────
m_full_naive = np.zeros(N_PATCHES)
m_full_naive[int_mask] = m_naive

ext = [patch_lon.min()-0.5, patch_lon.max()+0.5,
       patch_lat.min()-0.3, patch_lat.max()+0.3]

fig, ax = plt.subplots(figsize=(9, 9),
                        subplot_kw={'projection': PROJ},
                        constrained_layout=True)
setup_map(ax, extent=ext)
vmax = m_naive.max() * 100
if bnd_mask.sum():
    ax.tripcolor(coords[:,0], coords[:,1], tri_idx[bnd_mask],
                 facecolors=np.zeros(bnd_mask.sum()),
                 cmap='Greys', vmin=0, vmax=1, zorder=2, alpha=0.3)
tc = ax.tripcolor(coords[:,0], coords[:,1], tri_idx[int_mask],
                  facecolors=m_naive*100, cmap='hot_r',
                  vmin=0, vmax=vmax, zorder=3, alpha=0.9)
ax.triplot(coords[:,0], coords[:,1], tri_idx,
           color='0.5', lw=0.15, zorder=4, alpha=0.3)
plt.colorbar(tc, ax=ax, shrink=0.6, pad=0.02, label='Slip (cm)')
for i, sta in enumerate(sta_list):
    v = offsets_inv[sta]
    ax.quiver(v['lon'], v['lat'],
              v['de']*ARROW_SCALE, v['dn']*ARROW_SCALE,
              color='steelblue', scale=1, scale_units='xy', angles='xy',
              width=0.005, headwidth=4, zorder=6, transform=PROJ,
              label='Observed' if i==0 else '_')
    ax.quiver(v['lon'], v['lat'],
              d_pred_naive[2*i]*ARROW_SCALE, d_pred_naive[2*i+1]*ARROW_SCALE,
              color='firebrick', scale=1, scale_units='xy', angles='xy',
              width=0.005, headwidth=4, zorder=6, transform=PROJ,
              label='Predicted' if i==0 else '_')
ax.legend(fontsize=9, loc='lower left')
ax.set_title(f'Slip distribution  (λ = {lam_opt:.2f})\n'
             f'Peak = {m_naive.max()*100:.1f} cm   '
             f'NRMS = {nrms_naive:.2f}   Mw = {Mw_naive:.2f}', fontsize=10)
plt.show()


#### Exercise: Can you regularize your way to a reasonable moment?

Go back to `iv2_code` and change `LAM_OPT` to each of the values below.
Re-run `iv2_code` then `iv3_code` and record the results in this table:

| λ | NRMS | Peak slip (cm) | Mw |
|---|------|---------------|----|
| 0.1 | | | |
| 1   | | | |
| 10  | | | |
| 100 | | | |

> **Think about it:** As you increase λ, the slip distribution becomes smoother
> and NRMS increases (worse fit). Does Mw decrease substantially?
> Can you find a value of λ that simultaneously gives a good fit *and* a
> physically reasonable moment magnitude (Mw 6.0–6.5)?


### IV.4 — Boundary constraints and amplitude damping

From the exercise above you found that Mw stays roughly constant no matter
how large you make λ. This is not a numerical accident.

**Why smoothing cannot constrain total moment.**
The Laplacian penalty measures how much slip *varies between neighbours*.
A solution where every patch slips equally has zero smoothing cost regardless
of that common level. The smoothing operator has no opinion about the *average*
slip across the mesh — only its spatial gradient. Increasing λ spreads moment
more uniformly over a large area, which is also smooth, so Mw barely changes.

**Two additional constraints fix this.**

The complete augmented system solved by NNLS is:

$$\hat{\mathbf{m}} = \underset{\mathbf{m}\,\geq\,0}{\operatorname{argmin}}
\left\|\begin{bmatrix}
W_d G \\
\lambda\, L_\text{int} \\
\lambda_\text{damp}\, I
\end{bmatrix}\mathbf{m}
- \begin{bmatrix} W_d\mathbf{d} \\ \mathbf{0} \\ \mathbf{0}\end{bmatrix}
\right\|^2$$

Each row of the augmented system enforces one constraint:

| Row |    Term    | Purpose | Code parameter |
|-----|------------|---------|---------------|
| 1 | $W_d G\,\mathbf{m} \approx W_d\mathbf{d}$ | Fit the GNSS data (least squares) | — |
| 2 | $\lambda\, L_\text{int}\,\mathbf{m} \approx \mathbf{0}$ | Spatial smoothness; $L_\text{int}$ also forces slip to taper to zero at the mesh boundary (Dirichlet BCs) | `LAM_OPT` |
| 3 | $\lambda_\text{damp}\, I\,\mathbf{m} \approx \mathbf{0}$ | Directly penalises slip amplitude on every patch | `LAM_DAMP` |
| + | $\mathbf{m} \geq 0$ | Non-negative (thrust) slip only | NNLS solver |

**Why $L_\text{int}$ differs from the standard $L$.**
The standard Laplacian is built only from interior patches, so uniform slip
across all interior patches costs nothing. $L_\text{int}$ is the sub-matrix
of the full-mesh Laplacian: for an interior patch adjacent to a boundary patch, its
diagonal entry still counts the fixed-zero boundary neighbour. This means the
row-2 penalty is non-zero even for uniform interior slip, and the inversion is
pushed to taper slip toward zero at the edges — a physically motivated constraint
since the mesh boundary is a computational artefact, not a real slip boundary.

Adjust the three parameters below and observe their effects.


In [ ]:
# Full-mesh Laplacian → interior sub-matrix (Dirichlet BCs)
cx_all, cy_all = lonlat_to_m(patch_lon, patch_lat)
cents_all_m    = np.column_stack([cx_all, cy_all])
L_full         = build_spatial_laplacian(cents_all_m, LAP_RADIUS_KM * 1e3)
L_int    = L_full[np.ix_(int_mask, int_mask)]

# ── Tuning parameters — adjust and re-run to see effects ─────────────────────
LAM_DAMP  = 1.0    # amplitude damping weight (0 = none)
LAM_OPT = 10.0   # smoothing λ; set None to use L-curve auto-corner

# ── L-curve (run once; re-run if LAM_DAMP changes) ───────────────────────────
print('L-curve with L_int …')
nrms_d, rough_d, m_list_d = run_lcurve(
    G, d_obs, sig_d, LAMBDAS, L_int, lam_damp=LAM_DAMP)
corner_d = lcurve_corner(nrms_d, rough_d)
lam_opt  = LAM_OPT if LAM_OPT is not None else LAMBDAS[corner_d]
print(f'  Auto corner: λ = {LAMBDAS[corner_d]:.2f}   Using: λ = {lam_opt:.2f}')

# ── Solve ─────────────────────────────────────────────────────────────────────
rows = [G * w[:,None], lam_opt * L_int]
rhs  = [d_obs * w,     np.zeros(N_INT)]
if LAM_DAMP > 0:
    rows.append(LAM_DAMP * np.eye(N_INT)); rhs.append(np.zeros(N_INT))
m_dir, _ = nnls(np.vstack(rows), np.concatenate(rhs))

d_pred_dir = G @ m_dir
nrms_dir   = float(np.sqrt(np.mean(((d_obs - d_pred_dir) / sig_d)**2)))
M0_dir, Mw_dir = compute_mw(m_dir, int_areas)

print(f'\n{"":30s}  {"Standard":>12s}  {"Dirichlet+damp":>14s}')
print(f'{"NRMS":30s}  {nrms_naive:12.2f}  {nrms_dir:14.2f}')
print(f'{"Peak slip (cm)":30s}  {m_naive.max()*100:12.1f}  {m_dir.max()*100:14.1f}')
print(f'{"Mw":30s}  {Mw_naive:12.2f}  {Mw_dir:14.2f}')

# ── Side-by-side slip maps with observed and predicted arrows ─────────────────
vmax2 = max(m_naive.max(), m_dir.max()) * 100
preds = [d_pred_naive, d_pred_dir]
titles = [f'Standard (L)\nMw = {Mw_naive:.2f}  NRMS = {nrms_naive:.2f}',
          f'L_int + damping\nMw = {Mw_dir:.2f}  NRMS = {nrms_dir:.2f}']
slips = [m_naive, m_dir]

fig = plt.figure(figsize=(14, 9), constrained_layout=True)
for col, (title, slip, d_pred) in enumerate(zip(titles, slips, preds)):
    ax = fig.add_subplot(1, 2, col+1, projection=PROJ)
    setup_map(ax, extent=ext)
    if bnd_mask.sum():
        ax.tripcolor(coords[:,0], coords[:,1], tri_idx[bnd_mask],
                     facecolors=np.zeros(bnd_mask.sum()),
                     cmap='Greys', vmin=0, vmax=1, zorder=2, alpha=0.3)
    tc = ax.tripcolor(coords[:,0], coords[:,1], tri_idx[int_mask],
                      facecolors=slip*100, cmap='hot_r',
                      vmin=0, vmax=vmax2, zorder=3, alpha=0.9)
    ax.triplot(coords[:,0], coords[:,1], tri_idx,
               color='0.5', lw=0.12, zorder=4, alpha=0.25)
    for i, sta in enumerate(sta_list):
        v = offsets_inv[sta]
        ax.quiver(v['lon'], v['lat'],
                  v['de']*ARROW_SCALE, v['dn']*ARROW_SCALE,
                  color='steelblue', scale=1, scale_units='xy', angles='xy',
                  width=0.005, headwidth=4, zorder=6, transform=PROJ,
                  label='Observed' if i == 0 else '_')
        ax.quiver(v['lon'], v['lat'],
                  d_pred[2*i]*ARROW_SCALE, d_pred[2*i+1]*ARROW_SCALE,
                  color='firebrick', scale=1, scale_units='xy', angles='xy',
                  width=0.005, headwidth=4, zorder=6, transform=PROJ,
                  label='Predicted' if i == 0 else '_')
    if col == 0:
        ax.legend(fontsize=8, loc='lower left')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Longitude')
fig.axes[0].set_ylabel('Latitude')
fig.colorbar(tc, ax=fig.axes, shrink=0.52, pad=0.02, label='Slip (cm)')
plt.suptitle('Slip inversion comparison', fontsize=12)
plt.show()


#### Exercise: explore boundary constraints and amplitude damping

Change `LAM_OPT` and `LAM_DAMP` at the top of `iv4_code` and re-run it.
Record results below. The goal is to find settings that bring Mw into
the published range (6.0–6.5) **without** letting NRMS exceed 2.0 —
above that threshold you are underfitting real signal, not just noise.

| `LAM_OPT` | `LAM_DAMP` | NRMS | Peak slip (cm) | Mw | In range? |
|-----------|-----------|------|---------------|-----|----------|
| 10 | 0 | | | | |
| 10 | 1 | | | | |
| 10 | 10 | | | | |
| 10 | 50 | | | | |
| 1  | 1  | | | | |
| 100| 1  | | | | |

> **Challenge:** Can you find a combination of `LAM_OPT` and `LAM_DAMP` that
> gives Mw ≤ 6.5 while keeping NRMS ≤ 2.0? If you cannot, what does that
> tell you about whether the high moment is a regularisation artifact or
> a genuine feature of the data?

> 1. With `LAM_DAMP = 0`, does the slip distribution look cleaner or messier?
>    What does this tell you about the role of amplitude damping?
> 2. What happens to the predicted arrows (red) relative to the observed (blue)
>    as you increase `LAM_DAMP` well above 1? Is that a good trade-off?
> 3. Try `LAM_OPT = 10, LAM_DAMP = 50`. Check the NRMS. Would you trust
>    this result? Why or why not?


### IV.5 — Source parameters

Compute the full source characterisation for the preferred (Dirichlet) solution.
The equivalent moment magnitude can be compared to published values for
northern Cascadia ETS events to assess whether the result is physically
reasonable.


In [ ]:
print('=== Source parameters: preferred solution (Dirichlet BCs) ===')
print(f'  Peak slip:          {m_dir.max()*100:.1f} cm')
print(f'  Mean slip (interior): {m_dir.mean()*10:.1f} mm')
print(f'  Total fault area:   {int_areas.sum()/1e9:.0f} × 10³ km²')
print(f'  Seismic moment M₀:  {M0_dir:.3e} N·m')
print(f'  Moment magnitude:   Mw {Mw_dir:.2f}')
print(f'  Event duration:     {(SSE_END - SSE_START).days} days')
print()
print('Reference — published Cascadia ETS Mw (Michel et al. 2019): 5.5 – 6.5')
print(f'This result is Mw {Mw_dir:.2f}, which is '
      f'{"within" if 6.0 <= Mw_dir <= 6.5 else "outside"} the published range.')


### IV.6 — Scaling context: event duration versus magnitude

Michel et al. (2019) compiled Cascadia ETS events and showed that event
duration and seismic moment scale predictably — longer events release more
moment. Plot your result alongside approximate published values to see
whether a ~30-day event at your estimated Mw is consistent with the
observed scaling.


In [ ]:
# Approximate published Cascadia ETS data
# (compiled from Michel et al. 2019, Bartlow 2020, and references therein;
#  values are illustrative — refer to the original papers for precise numbers)
pub_dur = np.array([ 3,  5,  7,  8, 10, 12, 14, 18, 20, 25, 28, 30, 35, 42, 50, 60])
pub_mw  = np.array([5.4, 5.6, 5.7, 5.8, 6.0, 6.0, 6.1, 6.2, 6.3, 6.3,
                     6.4, 6.4, 6.5, 6.5, 6.7, 6.8])

# Your result
event_duration = (SSE_END - SSE_START).days
event_Mw       = Mw_dir   # constrained (L_int + damping) solution

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(pub_dur, pub_mw, s=60, color='steelblue', alpha=0.75, zorder=3,
           label='Published Cascadia ETS (Michel et al. 2019 / Bartlow 2020)')

# Simple log-linear fit to published data
fit = np.polyfit(np.log10(pub_dur), pub_mw, 1)
x_fit = np.logspace(np.log10(2), np.log10(80), 100)
ax.plot(x_fit, np.polyval(fit, np.log10(x_fit)),
        color='steelblue', lw=1.5, ls='--', alpha=0.6, label='Scaling trend')

# Your event
ax.scatter(event_duration, event_Mw, s=200, color='firebrick',
           marker='*', zorder=5,
           label=f'Your result: {event_duration} days, Mw {event_Mw:.2f}')

ax.set_xscale('log')
ax.set_xlabel('Event duration (days)')
ax.set_ylabel('Moment magnitude Mw')
ax.set_title('ETS duration vs moment magnitude\n'
             'Cascadia subduction zone')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

print(f'Published scaling prediction for {event_duration} days: '
      f'Mw ≈ {np.polyval(fit, np.log10(event_duration)):.2f}')
print(f'Your result:                              Mw = {event_Mw:.2f}')


> **Interpret the inversion results:**
> 1. Where is slip concentrated in the preferred solution? Does this match
>    the tremor distribution you mapped in Part I?
> 2. The standard inversion (L) gives a higher Mw than the Dirichlet
>    solution. Looking at the two slip maps, where does the extra moment come
>    from — is it higher peak slip, or more broadly distributed background slip?


---

## Synthesis

Answer each question in 3–5 sentences.

> **1.** You identified the 2024 ETS using seismic tremor and confirmed it with GNSS.
> These are two independent datasets. What additional confidence does that give
> compared to using either alone? Describe one situation where tremor and GNSS
> might give inconsistent pictures of the same event.

> **2.** The standard inversion (Part IV.3) produces a higher Mw than the
> constrained inversion (Part IV.4). The difference is not because the data
> changed — the same G and d are used. Explain in physical terms why adding
> boundary constraints and amplitude damping reduces the inferred moment.

> **3.** Compare the spatial pattern of slip in your preferred inversion to
> the tremor distribution you mapped in Part I. Where do the two agree?
> Where do they disagree, and what could explain any mismatch?

> **4.** You chose a regularisation parameter λ using the L-curve. Explain
> to a non-expert why the solution changes as λ varies and what criterion
> you used to settle on a specific value. What would an NRMS much less
> than 1 suggest about your data uncertainties?

> **5.** The duration–magnitude plot in IV.6 places your result on the
> published Cascadia ETS scaling trend. Is your Mw consistent with the
> scaling prediction for a ~30-day event? If your result lies off the trend,
> discuss whether the discrepancy is more likely from the inversion setup
> (regularisation, boundary constraints) or from the GNSS data processing
> (window length, CMF treatment).


---

## Summary

**Part I — Tremor**
- ETS events appear in the tremor catalog as elevated detection rates lasting
  days to weeks against a low inter-event background, recurring on a ~14-month
  cycle in northern Cascadia.
- The onset of an ETS is typically sharper and easier to define than the end;
  start and end dates carry an uncertainty of a few days.

**Part II — GNSS offsets**
- GNSS offsets are measured as the difference between post- and pre-event means
  of the secular residual; propagating daily position uncertainties gives
  per-station displacement uncertainties.
- A common-mode filter removes signals shared across all network stations by
  computing the median offset of far-field reference stations (> 600 km from
  the event) and subtracting it from all analysis stations.

**Part III — Green's functions**
- The Green's function matrix G relates unit slip on each fault patch to
  surface displacement at each GNSS station. Each column of G is computed
  using the elastic half-space solution (cutde) for updip dip-slip.
- The sign of G can be verified: updip slip on the deep Cascadia interface
  should produce westward (negative east) motion at coastal stations.

**Part IV — Slip inversion**
- The regularised inverse problem minimises data misfit plus a Laplacian
  smoothness penalty and an amplitude damping term, subject to non-negative
  (thrust) slip.
- The standard Laplacian does not constrain the average slip level across
  the mesh; moment inflates because smoothing cannot distinguish a localised
  ETS patch from broadly distributed background slip.
- Replacing the standard Laplacian with the Dirichlet sub-matrix forces slip
  to taper to zero at the mesh boundary, concentrating moment in the interior
  and bringing the estimated Mw into the published Cascadia ETS range (6.0–6.5).
- The inferred slip distribution peaks at 25–45 km depth, consistent with the
  ETS seismogenic transition zone, and the event duration and magnitude are
  consistent with the Cascadia ETS scaling relation.

### Data sources

- **PNSN tremor catalog**: Pacific Northwest Seismic Network. Record the
  commit hash used to access the dataset.
- **GNSS time series**: SOPAC compilation distributed through the course
  repository. Record the commit hash, processing version, and reference
  frame when reporting results.
- **Fault mesh**: McCrory et al. (2012) Cascadia interface geometry,
  triangulated and distributed through the course repository.
